In [1]:
import kagglehub
import pandas as pd
import os

c:\Users\aryan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
#Download the dataset
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print("Path to dataset files:", path)

# Load the dataset into a dataframe
data = pd.read_csv(path + "/twcs/twcs.csv")

Path to dataset files: C:\Users\aryan\.cache\kagglehub\datasets\thoughtvector\customer-support-on-twitter\versions\10


In [47]:
data.head(20)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0


From the data I have understood that

1. Inbound refers to tweets that was recieved by companies, hence True and outbond are those tweets which are tweeted by companies in response to users, which is True.

2. If in_response_to_tweet_id is Nan, that means those were the root threads.

3. If response_tweet_id is Nan, that means those were the last threads in the conversation.

4. Inorder to make underatsnding more eay, we need to map all those threads together.

In [48]:
print("Shape of the data is: ", data.shape)
print("Columns in the data are: ", data.columns.tolist())
print(data.info())

Shape of the data is:  (2811774, 7)
Columns in the data are:  ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB
None


In [50]:
data["created_at"] = pd.to_datetime(data['created_at'])

In [52]:
# Find the number of times author_id appears in the data
print("Number of times author_id appears in the data is: ", data['author_id'].value_counts())

Number of times author_id appears in the data is:  author_id
AmazonHelp      169840
AppleSupport    106860
Uber_Support     56270
SpotifyCares     43265
Delta            42253
                 ...  
456282               1
456281               1
456280               1
456276               1
823870               1
Name: count, Length: 702777, dtype: int64


As AmazonHelp has the highest volume of tweets, I decided with go with the brand "AmazonHelp"

Also these are just the tweets that have been sent by AmazonHelp, we need to get all the tweets in the thread for a better context.

In [53]:
tweet_index = data.set_index('tweet_id')

roots = data[data['in_response_to_tweet_id'].isna()]['tweet_id'].tolist()

def collectThreads(rootId):
    threadTweets = []
    frontier = [rootId]
    seen = set()
    while frontier:
        nextFrontier = []

        for tid in frontier:
            if tid in seen or tid not in tweet_index.index:
                continue

            seen.add(tid)

            row = tweet_index.loc[tid]
            threadTweets.append({
                "tweet_id": tid,
                "author_id": row['author_id'],
                "inbound": row['inbound'],
                "text": row["text"],
                "created_at": row["created_at"],
            })
            children = row["response_tweet_id"]
            if pd.notna(children):
                nextFrontier = nextFrontier + [int(x) for x in children.split(",")]
        frontier = nextFrontier
    threadTweets.sort(key=lambda x: x["created_at"])
    return threadTweets

amazonThreads = []
for rootId in roots:
    tweets = collectThreads(rootId)
    root_row = tweet_index.loc[rootId]
    if any(t['author_id'] == 'AmazonHelp' for t in tweets):
        amazonThreads.append({"thread_id": str(rootId), "thread_text": root_row["text"], "tweets": tweets})

print(f"{len(amazonThreads)} Amazon threads found in sample of {len(roots)} roots")


81902 Amazon threads found in sample of 794335 roots


In [54]:
amazonThreads

[{'thread_id': '272',
  'thread_text': 'amazonのfireTVstickが見れない😢',
  'tweets': [{'tweet_id': 272,
    'author_id': '115770',
    'inbound': True,
    'text': 'amazonのfireTVstickが見れない😢',
    'created_at': Timestamp('2017-11-22 09:14:39+0000', tz='UTC')},
   {'tweet_id': 269,
    'author_id': 'AmazonHelp',
    'inbound': False,
    'text': '@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET',
    'created_at': Timestamp('2017-11-22 09:23:01+0000', tz='UTC')},
   {'tweet_id': 270,
    'author_id': '115770',
    'inbound': True,
    'text': '@AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。',
    'created_at': Timestamp('2017-11-22 09:24:30+0000', tz='UTC')},
   {'tweet_id': 271,
    'author_id': '115770',
    'inbound': True,
    'text': '@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。',
    'created_at': Timestamp('2017-11-22 09:30:36+0000', tz='UTC')},
   {'tweet_id': 273,
    'author_id': 'AmazonHe